# CRISP-ML(Q) Phase 2 – Data Engineering

**Goal:** turn ~3.8 million raw SINAN records into clean, balanced modeling tables, with the same feature definitions the
web app uses.

Code: `src/sinan.py` (selection, labels, sampling) and `src/preprocessing.py` (feature definitions and decoding).

Previous phase: `01_business_and_data_understanding.ipynb` · Next phase: `03_model_engineering.ipynb`

In [1]:
import sys
sys.path.insert(0, "..")  # project root, to import the src package

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)

import inspect
from src import preprocessing, sinan
from src.preprocessing import BINARY_GROUPS, CLASS_LABELS, CLASSES, FEATURE_LABELS, FEATURES
from src.sinan import TRAIN_YEAR, VALIDATION_YEAR, load_dataset

## 2.1 Target variable

| Class | Rule (laboratory criterion only, `CRITERIO = 1`) |
|---|---|
| `dengue` | `CLASSI_FIN` 10, 11 or 12 |
| `chikungunya` | `CLASSI_FIN` 13 |
| `other_febrile` | `CLASSI_FIN` 5 (discarded) |

In [2]:
print(sinan.LABEL_CODES)

{'10': 'dengue', '11': 'dengue', '12': 'dengue', '13': 'chikungunya', '5': 'other_febrile'}


## 2.2 Features

All features are numeric, so tree models need no encoding:

In [3]:
rows = [("age", "derived", FEATURE_LABELS["age"]), ("days_since_onset", "derived", FEATURE_LABELS["days_since_onset"]),
        ("sex_female", "derived", FEATURE_LABELS["sex_female"]),
        ("state", "epidemiological context · SG_UF", FEATURE_LABELS["state"]),
        ("month", "epidemiological context · DT_NOTIFIC", FEATURE_LABELS["month"])]
for group, items in BINARY_GROUPS.items():
    rows += [(key, f"{group} · {column}", label) for key, (column, label) in items.items()]
pd.DataFrame(rows, columns=["feature", "group · SINAN column", "label"])

,feature,group · SINAN column,label
0,age,derived,Age
1,days_since_onset,derived,Days since symptom onset
2,sex_female,derived,Female sex
3,state,epidemiological context · SG_UF,State of residence
4,month,epidemiological context · DT_NOTIFIC,Month of consultation
5,fever,Symptoms · FEBRE,Fever
6,headache,Symptoms · CEFALEIA,Headache
7,retro_orbital_pain,Symptoms · DOR_RETRO,Pain behind the eyes
8,myalgia,Symptoms · MIALGIA,Muscle pain
9,arthralgia,Symptoms · ARTRALGIA,Joint pain


**Epidemiological context.** The first version used only clinical features and reached a macro ROC-AUC of 0.71. Adding the
patient's **state of residence** and the **month** (local circulation, the pre-test probability a clinician uses) raised it
above 0.80. The state is the IBGE code as a number; tree models split it into groups of states.

Decoding rules (`src/preprocessing.py`):
- Yes/no fields: `1 = Yes → 1`, `2 = No → 0`, anything else → missing.
- Age: `NU_IDADE_N` = unit digit + value (4 = years, 3 = months, 2 = days, 1 = hours). Outside 0–110 → missing.
- Days since onset = notification date − first-symptom date. Outside 0–30 → missing.

In [4]:
print(inspect.getsource(preprocessing.age_in_years))
print(inspect.getsource(sinan.to_features))

def age_in_years(series):
    """Decodes SINAN's NU_IDADE_N: the first digit is the unit (1 hours, 2 days, 3 months, 4 years)."""
    text = series.astype(str).str.strip()
    unit, value = text.str[:1], pd.to_numeric(text.str[1:], errors="coerce")
    years = value.where(unit == "4", (value / 12).where(unit == "3", (value / 365).where(unit == "2", 0)))
    return years.where(unit.isin(["1", "2", "3", "4"])).where(lambda a: a.between(0, MAX_AGE))

def to_features(raw):
    """Raw SINAN records -> model features (see src/preprocessing.py) + label and context columns."""
    X = pd.DataFrame(index=raw.index)
    X["age"] = age_in_years(raw["NU_IDADE_N"])
    X["days_since_onset"] = days_between(raw["DT_SIN_PRI"], raw["DT_NOTIFIC"])
    X["sex_female"] = raw["CS_SEXO"].map({"F": 1.0, "M": 0.0})
    X["notified"] = pd.to_datetime(raw["DT_NOTIFIC"], format="%Y%m%d", errors="coerce")
    X["state"] = pd.to_numeric(raw["SG_UF"], errors="coerce")  # epidemiological context: where
    X["month"

## 2.3 Balanced samples

Laboratory-confirmed dengue is much more frequent than the other two classes. An **equal number of cases per class** is
sampled, so the model does not learn "always answer dengue" and its probabilities are comparable across diagnoses.
Records with missing age, sex or any yes/no field are dropped (very few).

In [5]:
for year in (TRAIN_YEAR, VALIDATION_YEAR):
    s = json.loads(open(f"../data/processed/arbovirus_{year}_summary.json", encoding="utf-8").read())
    print(f"{year}: laboratory-confirmed {s['lab_confirmed']} | dropped incomplete {s['dropped_incomplete']:,} | sample {s['sample']}")

2023: laboratory-confirmed {'dengue': 654045, 'chikungunya': 56197, 'other_febrile': 58353} | dropped incomplete 1,126 | sample {'dengue': 40000, 'chikungunya': 40000, 'other_febrile': 40000}
2025: laboratory-confirmed {'dengue': 511555, 'chikungunya': 64283, 'other_febrile': 85045} | dropped incomplete 396 | sample {'dengue': 15000, 'chikungunya': 15000, 'other_febrile': 15000}


In [6]:
X, y, context = load_dataset(TRAIN_YEAR)
print(X.shape)
X.describe().T[["mean", "min", "max"]]

(120000, 26)


,mean,min,max
age,38.200,0.0,109.0
days_since_onset,4.455,0.0,30.0
sex_female,0.580,0.0,1.0
state,33.585,11.0,53.0
month,4.843,1.0,12.0
fever,0.797,0.0,1.0
headache,0.707,0.0,1.0
retro_orbital_pain,0.241,0.0,1.0
myalgia,0.737,0.0,1.0
arthralgia,0.352,0.0,1.0


## 2.4 Phase output

| File | Use |
|---|---|
| `data/processed/arbovirus_2023.parquet` | Training and test (Phases 3–4) |
| `data/processed/arbovirus_2025.parquet` | Temporal validation (Phase 4) |
| `src/tulua.py` | Colombian data mapped to the same features (external validation) |

Next: `03_model_engineering.ipynb`.